# 🏥 TRICARE RAG Q&A 통합 파이프라인 v5 (integrated5 → v3)

## 개요
## [integrated5 → v3] 변경 사항
| 위치 | 기존 | 변경 |
|---|---|---|
| Section 0 | 패키지 목록 | `rank-bm25`, `sentence-transformers` 추가 |
| Section 10 | MMR 단일 검색 | BM25 + Hybrid + CrossEncoder Rerank 파이프라인 |
| Section 11 | make_rag_chain_v2까지 | `make_rag_chain_v3` 추가 (Hybrid+Rerank 통합) |
| Demo 1, 2 | make_rag_chain (MMR) | make_rag_chain_v3 (Hybrid+Rerank) |

**반영 근거**: `tricare_retriever_exp.ipynb` EX-5 정량 평가
- D (Hybrid+Rerank): Hit Rate 1.0 / MRR 0.867 / Faithfulness 1.0
- A (MMR 기존): Hit Rate 1.0 / MRR 0.667 / Faithfulness 0.920

- `tricare_rag_integrated4.ipynb` 기반으로 전처리·검색 방식 개선한 버전임
- 신규 PDF 3종은 다음 창에서 추가 예정: `RAG_PDF_DOCS` 리스트와 `TABLE_PDF_DOCS`에 경로만 추가하면 됨

## 변경 사항 (integrated4 대비)

| Section | 변경 내용 |
|---|---|
| **Section 4** | `clean_text()` · `normalize_cell()` · `enrich_tricare_text()` · `is_noise_line()` 추가 |
| **Section 5** | RecursiveCharacterTextSplitter → **단락(`\\n\\n`) 단위 직접 분리** (80자 미만 제거) |
| **Section 8** | 표 셀에 `normalize_cell()` 적용, 빈 행 제거 |
| **Section 10** | `similarity` → **MMR** 검색, `normalize_question()` 추가 |
| **Section 11** | Term Locking 프롬프트 추가, `make_rag_chain_v2()` 추가 (MMR 완전 통합) |


```

## 팀별 벡터 DB 경로 (통합 시 참고)
| 팀원 | persist_directory | collection_name |
|------|------------------|-----------------|
| TRICARE  | `./chroma_db`, `./chroma_db2` | `tricare_rag`, `tricare_cost_tables` |
| Cigna (은우님) | `./chroma_cigna_latest`, `./chroma_cigna_all` | `cigna_latest`, `cigna_all` |
| Allianz (수진님) | `./vectordb` | `allianz_care` |

## 전체 실행 순서
1. 패키지 설치 → 환경 설정 → 파일 경로 설정
2. 임베딩 모델 초기화
3. 전처리 유틸 함수 정의 (clean_text, normalize_cell, enrich_tricare_text, is_noise_line) → PDF 로드
4. 단락 단위 청킹 (RecursiveCharacterTextSplitter는 CSV에만 사용)
5. CSV 로드 → Document 변환 → 청킹
6. 텍스트+CSV 통합 벡터 저장소 생성 (`chroma_db`)
7. normalize_cell 적용 PDF 표 추출 + Health_Plan_Costs.csv → 표 전용 벡터 저장소 생성 (`chroma_db2`)
8. MMR Retriever + normalize_question 설정
9. LLM + RAG 체인 구성 (Term Locking 프롬프트 추가)
10. Demo 실행


---
## Section 0 · 패키지 설치

### 개요
- 처음 실행 시에만 필요함
- 주석 해제 후 실행하면 됨


In [1]:
# [integrated5 → v3 변경] rank-bm25, sentence-transformers 추가
# - rank-bm25: BM25Retriever 사용에 필요
# - sentence-transformers: CrossEncoder Reranker 사용에 필요
# 처음 실행 시 아래 주석 해제 후 실행

# !pip install langchain langchain-community langchain-chroma langchain-openai \
# pymupdf pdfplumber sentence-transformers chromadb python-dotenv -q

# [신규 추가] BM25 + Reranker 패키지
# !pip install rank-bm25 sentence-transformers -q


---
## Section 1 · 환경 설정

### 개요
- `.env` 파일에서 `OPENAI_API_KEY`를 읽어옴
- 벡터 DB 경로·컬렉션명을 상수로 분리해 관리함
  - `PERSIST_TEXT` / `COLLECTION_TEXT`: 텍스트+CSV 통합 저장소
  - `PERSIST_TABLE` / `COLLECTION_TABLE`: 비용 표 전용 저장소
- 팀 통합 벡터 DB 병합 시 이 상수들을 기준으로 경로를 맞춤


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')
print(f"✅ API Key 로드: {'성공' if api_key else '실패 → .env 파일 확인 필요'}")

# 벡터 DB 경로·컬렉션명 상수
# 팀 통합 시 이 두 상수만 보면 어느 팀원 DB인지 바로 알 수 있도록 tricare_ 프리픽스 유지
PERSIST_TEXT = './chroma_db'
PERSIST_TABLE= './chroma_db2'   # 표 내용도 들어감
COLLECTION_TEXT  = 'tricare_rag'
COLLECTION_TABLE = 'tricare_cost_tables'


✅ API Key 로드: 성공


---
## Section 2 · 파일 경로 설정

### 개요
- [변경] 기존 `PDF_COSTS`, `PDF_HANDBOOK` 개별 변수 방식 → `RAG_PDF_DOCS` 리스트 방식으로 교체함
  - 신규 PDF가 추가될 때 리스트에 경로 한 줄만 추가하면 되는 구조임
  - 유지보수성이 크게 향상됨
- `RAG_PDF_DOCS`: `chroma_db`(텍스트 저장소)에 들어갈 PDF 목록
- `TABLE_PDF_DOCS`: `chroma_db2`(표 전용 저장소)에 추가로 표 추출할 PDF 목록
  - `Costs_Fees.pdf`는 두 리스트 모두에 포함됨 (텍스트 + 표 이중 수록 의도적)
  - `TRICARE_ADDP_HB.pdf`는 비용 표가 없으므로 `TABLE_PDF_DOCS`에서 제외함
- 경로는 본인 로컬 환경에 맞게 수정할 것

### 신규 추가 파일 (integrated2 대비)
| 파일 | 저장소 | 이유 |
|---|---|---|
| `Overseas_HB.pdf` | RAG만 | 해외 절차·한국 Near Patient Program 등 서술형 |
| `Pharmacy_HB.pdf` | RAG + TABLE | 본문 서술 + 비용표 모두 존재 |
| `NGR_HB.pdf` | RAG + TABLE | 본문 서술 + 활성화 상태별 플랜 옵션 표 |
| `TRICARE_ADDP_HB.pdf` | RAG만 | ADSM 비용 없는 프로그램, 비용 표 없음 |
| `ADDP_Brochure.pdf` | RAG만 | 개요 서술 |
| `Maternity_Br.pdf` | RAG만 | 서술형 브로슈어 |
| `Retiring_AD_Br.pdf` | RAG만 | 서술형 브로슈어 |
| `Retiring_NGR_Br.pdf` | RAG만 | 서술형 브로슈어 |
| `QLEs_FS.pdf` | RAG만 | 서술형 팩트시트 |

In [ ]:
import os

# 경로 설정 
# DATA_DIR: 로컬 PC의 PDF/CSV 파일이 들어있는 폴더 경로
# 이 변수 하나만 바꾸면 아래 모든 경로가 자동으로 바뀜
DATA_DIR = r'C:\SKN_24\3차 프로젝트\Care-Insure-Bot\data'

# RAG_PDF_DOCS
# chroma_db(tricare_rag)에 텍스트로 들어갈 PDF 목록
# 한국/OCONUS 적용 가능한 문서만 포함
# BOTH = 미국·해외 공통 / OCONUS = 해외 전용
# (CONUS 전용 문서 제외: Choices_US_HB, Retiring_AD_Br)
RAG_PDF_DOCS = [
    # 비용 (BOTH) 
    os.path.join(DATA_DIR, 'Costs_Fees.pdf'),

    # 해외 전용 핸드북 (OCONUS)
    os.path.join(DATA_DIR, 'Overseas_HB(해외 프로그램 안내서).pdf'),

    # 약국 (BOTH: 해외 약국 섹션 포함)
    os.path.join(DATA_DIR, 'Pharmacy_HB(tricare 약국 프로그램 안내서).pdf'),

    #  예비군·방위군 (BOTH: 활성화 시 해외 적용) 
    os.path.join(DATA_DIR, 'NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf'),

    #  평생 트라이케어 (BOTH: 해외 TFL 규정 포함) 
    os.path.join(DATA_DIR, 'TFL_HB(평생 트라이케어).pdf'),

    #  치과 (BOTH: OCONUS ADDP 섹션 포함) 
    os.path.join(DATA_DIR, 'TRICARE_ADDP_HB_FINAL_508c(현역 군인 치과 프로그램 안내서).pdf'),
    os.path.join(DATA_DIR, 'ADDP_Brochure_FINAL_122624_508c.pdf'),

    #  임산부 (BOTH: 해외 출산 규정 포함) ──
    os.path.join(DATA_DIR, 'Maternity_Br (1).pdf'),

    #  Medicare 전환 (BOTH: 해외 거주자 Medicare 규정 포함) -> 제외
    #os.path.join(DATA_DIR, 'Medicare_Turning_65_Br.pdf'),
    #os.path.join(DATA_DIR, 'Medicare_Under_65_Br (7).pdf'),

    #  예비군 전역 (BOTH: TRR→TFL 전환 포함) 
    os.path.join(DATA_DIR, 'Retiring_NGR_Br.pdf'),

    #  생애사건 (BOTH: 해외 이사·command sponsorship 포함) 
    os.path.join(DATA_DIR, 'QLEs_FS (2).pdf'),
]

#  TABLE_PDF_DOCS 
# chroma_db2(tricare_cost_tables)에 표(pdfplumber)로 들어갈 PDF 목록
# 실제 비용 수치 표가 있는 파일만 포함
TABLE_PDF_DOCS = [
    os.path.join(DATA_DIR, 'Costs_Fees.pdf'),
    os.path.join(DATA_DIR, 'Pharmacy_HB(tricare 약국 프로그램 안내서).pdf'),
    os.path.join(DATA_DIR, 'NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf'),
    os.path.join(DATA_DIR, 'TFL_HB(평생 트라이케어).pdf'),
]
#  CSV 경로 
CSV_MENTAL     = os.path.join(DATA_DIR, 'mental_health_services.csv')
CSV_COSTS      = os.path.join(DATA_DIR, 'Health_Plan_Costs.csv')
CSV_PLANS      = os.path.join(DATA_DIR, 'TricarePlans.csv')
CSV_EXCLUSIONS = os.path.join(DATA_DIR, 'tricare_exclusions.csv')

# ── 파일 존재 여부 확인 ──
print('[ RAG PDF 목록 ]')
for path in RAG_PDF_DOCS:
    status = '✅' if os.path.exists(path) else '❌ 없음 →'
    print(f'  {status} {os.path.basename(path)}')

print('\n[ TABLE PDF 목록 ]')
for path in TABLE_PDF_DOCS:
    status = '✅' if os.path.exists(path) else '❌ 없음 →'
    print(f'  {status} {os.path.basename(path)}')

print('\n[ CSV 목록 ]')
for path in [CSV_MENTAL, CSV_COSTS, CSV_PLANS, CSV_EXCLUSIONS]:
    status = '✅' if os.path.exists(path) else '❌ 없음 →'
    print(f'  {status} {os.path.basename(path)}')


[ RAG PDF 목록 ]
  ✅ Costs_Fees.pdf
  ✅ Overseas_HB(해외 프로그램 안내서).pdf
  ✅ Pharmacy_HB(tricare 약국 프로그램 안내서).pdf
  ✅ NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf
  ✅ TFL_HB(평생 트라이케어).pdf
  ✅ TRICARE_ADDP_HB_FINAL_508c(현역 군인 치과 프로그램 안내서).pdf
  ✅ ADDP_Brochure_FINAL_122624_508c.pdf
  ✅ Maternity_Br (1).pdf
  ✅ Retiring_NGR_Br.pdf
  ✅ QLEs_FS (2).pdf

[ TABLE PDF 목록 ]
  ✅ Costs_Fees.pdf
  ✅ Pharmacy_HB(tricare 약국 프로그램 안내서).pdf
  ✅ NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf
  ✅ TFL_HB(평생 트라이케어).pdf

[ CSV 목록 ]
  ✅ mental_health_services.csv
  ✅ Health_Plan_Costs.csv
  ✅ TricarePlans.csv
  ✅ tricare_exclusions.csv


---
## Section 3 · 임베딩 모델 초기화 — `BAAI/bge-m3`

### 개요
- 100개 이상의 언어를 동일한 벡터 공간에 매핑하는 다국어 임베딩 모델임
- 한국어 질문과 영어 문서 간 유사도 매칭이 가능함
- 팀원 Cigna 코드와 동일한 모델을 사용해 임베딩 방식을 통일함
- GPU 있으면 `device: cuda`로 변경하면 속도가 빨라짐


In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cpu'}  # GPU 있으면 'cuda'로 변경
)

print('✅ BAAI/bge-m3 임베딩 모델 로드 완료')


C:\Users\kjw70\AppData\Local\Temp\ipykernel_8328\2816061350.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(



✅ BAAI/bge-m3 임베딩 모델 로드 완료


---
## Section 4 · 전처리 유틸 함수 + PDF 로드

### 변경 사항
- `clean_text()`: PDF 특수문자·중복 공백 정제
- `normalize_cell()`: 표 체크마크 → Covered/Not covered 텍스트 변환
- `is_noise_line()`: 반복 헤더·URL 등 노이즈 행 제거
- `is_oconus_relevant()`: **[신규]** BOTH 문서에서 한국/해외 관련 페이지만 필터링
- `enrich_tricare_text()`: 본문 뒤에 search_tags 블록 추가
- `LOCATION_MAP`: 파일명 → OCONUS/BOTH 태그 매핑

### 페이지 필터링 로직
```
OCONUS 문서 → 전 페이지 사용 (이미 해외 전용)
BOTH 문서   → OCONUS_KEYWORDS 포함 페이지만 사용
CONUS 문서  → RAG_PDF_DOCS에서 이미 제외
```


In [5]:
#from langchain_community.document_loaders import PyMuPDFLoader
#
#all_pdf_docs = []
#
## RAG_PDF_DOCS 리스트를 순회하며 일괄 로드
## 신규 PDF가 추가되어도 Section 2의 리스트만 수정하면 이 셀은 수정 불필요
#for pdf_path in RAG_PDF_DOCS:
#    docs = PyMuPDFLoader(pdf_path).load()
#    # 파일명을 source_file 메타데이터로 추가해 팀 통합 시 출처 구분 용이하게 함
#    for doc in docs:
#        doc.metadata['source_file'] = os.path.basename(pdf_path)
#    all_pdf_docs.extend(docs)
#    print(f'  ✅ {os.path.basename(pdf_path)}: {len(docs)}페이지')
#
#print(f'\n📄 총 PDF 페이지 합계: {len(all_pdf_docs)}페이지')
#print(f'\n--- 샘플 메타데이터 ---')
#print(all_pdf_docs[0].metadata)


# 내용 수정: 일부 pdf - 보험 혜택 가이드라인 중 주한미군&한국 내 적용이 일부만 되는 것이 있기 때문

In [ ]:
import re
from langchain_community.document_loaders import PyMuPDFLoader


# 전처리 유틸 함수
def clean_text(text: str) -> str:
    """
    PDF에서 추출한 텍스트의 불필요한 문자 제거
    - \xa0 (non-breaking space): PDF에서 자주 나오는 특수 공백
    - \u200b (zero-width space): 눈에 안 보이지만 검색에 영향 주는 문자
    - 연속 공백·탭 → 공백 1개로 통일
    - 3줄 이상 빈 줄 → 2줄로 통일
    """
    text = text.replace('\xa0', ' ')
    text = text.replace('\u200b', ' ')
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def normalize_cell(cell: str) -> str:
    """
    표 셀의 체크마크·빈칸을 LLM이 읽기 좋은 텍스트로 변환
    - ✓/√/v/V/● → 'Covered'  (보장됨)
    - ✗/×/x/X/✘ → 'Not covered'  (보장 안 됨)
    - 빈칸        → 'N/A'
    """
    if not cell:
        return 'N/A'
    cell = cell.strip()
    if cell in ['✓', '√', 'v', 'V', '●', 'Yes', 'yes', 'Y']:
        return 'Covered'
    if cell in ['✗', '×', 'x', 'X', '✘', 'No', 'no', 'N']:
        return 'Not covered'
    return cell


def is_noise_line(line: str) -> bool:
    """
    반복 헤더·URL·업데이트 날짜 등 RAG에 불필요한 노이즈 행 판별
    True를 반환하면 해당 줄은 청크에서 제거됨
    """
    noise_patterns = [
        r'^group a.*group b',
        r'^covered service.*group',
        r'^are you in group',
        r'^this is an overview',
        r'^visit www\.tricare',
        r'^for more information.*go to',
        r'^updated (january|february|march|april|may|june|july|august|september|october|november|december)',
    ]
    return any(re.search(p, line.lower()) for p in noise_patterns)


# OCONUS 관련성 키워드
# BOTH 문서에서 이 키워드 중 하나라도 포함된 페이지만 남김
# 한국/해외 거주자에게 해당하는 내용만 파싱하기 위한 필터
OCONUS_KEYWORDS = [
    'overseas', 'oconus', 'outside the continental',
    'south korea', 'korea', 'usfk',
    'outside the u.s.', 'outside the united states',
    'international', 'host nation', 'foreign country',
    'tricare prime overseas', 'tricare select overseas',
    'tricare prime remote overseas',
    'near patient', 'overseas claim',
]

def is_oconus_relevant(text: str) -> bool:
    """
    페이지 텍스트에 해외/한국 관련 키워드가 있는지 확인
    - OCONUS 문서: 필터 없이 전 페이지 사용
    - BOTH 문서:   이 함수로 걸러서 관련 페이지만 사용
    - CONUS 문서:  RAG_PDF_DOCS에서 이미 제외됨
    """
    text_lower = text.lower()
    return any(kw in text_lower for kw in OCONUS_KEYWORDS)


# TRICARE_SEARCH_TAGS 
# 각 청크 본문 뒤에 붙이는 검색 향상용 키워드 블록
# bge-m3 임베딩에서 한국어 질문 → 영어 문서 유사도 매칭률을 높이는 역할
TRICARE_SEARCH_TAGS = [
    'coverage', 'covered', 'not covered', 'benefit', 'cost', 'copay',
    'deductible', 'enrollment fee', 'cost-share', 'prior authorization',
    'TRICARE Prime', 'TRICARE Select', 'TRICARE For Life',
    'Group A', 'Group B', 'active duty', 'retiree', 'reserve',
    'OCONUS', 'overseas', 'USFK', '주한미군',
    '보장', '비용', '본인부담금', '사전승인', '해외',
]


def enrich_tricare_text(text: str, fname: str) -> str:
    """
    본문 뒤에 [search_tags] 블록 추가
    - insurer: TRICARE → 팀 통합 시 어느 보험사 문서인지 구분
    - location: OCONUS/BOTH → 검색 필터링에 사용
    - keywords: 도메인 키워드 → 한/영 유사도 향상
    이 블록은 벡터 임베딩에는 포함되지만 LLM 컨텍스트에는 제거 후 전달됨
    (format_docs 함수에서 제거)
    """
    location = LOCATION_MAP.get(fname, 'BOTH')
    tags = '\n'.join([
        '[search_tags]',
        'insurer: TRICARE',
        f'location: {location}',
        'keywords: ' + ', '.join(TRICARE_SEARCH_TAGS),
    ])
    return f'{text}\n\n{tags}'


# LOCATION_MAP
# 파일명 → location 태그 매핑
# - OCONUS: 해외 전용 문서 → 전 페이지 사용
# - CONUS:  미국 내 전용 → RAG_PDF_DOCS에서 이미 제외됨
# - BOTH:   공통 문서 → OCONUS 키워드 있는 페이지만 사용
# 매핑에 없는 파일은 자동으로 'BOTH' 처리됨
LOCATION_MAP = {
    'Overseas_HB(해외 프로그램 안내서).pdf': 'OCONUS',
}
# 나머지 파일은 모두 BOTH (기본값)
# CONUS 전용 파일은 RAG_PDF_DOCS에서 이미 제외함


# PDF 로드 메인 루프
all_pdf_docs = []
skipped_pages = 0

for pdf_path in RAG_PDF_DOCS:
    docs  = PyMuPDFLoader(pdf_path).load()
    fname = os.path.basename(pdf_path)
    loc   = LOCATION_MAP.get(fname, 'BOTH')

    kept = []
    for doc in docs:
        cleaned = clean_text(doc.page_content)

        # OCONUS 문서: 전 페이지 사용
        # BOTH 문서:   OCONUS 키워드 있는 페이지만 사용
        if loc == 'BOTH' and not is_oconus_relevant(cleaned):
            skipped_pages += 1
            continue

        # 노이즈 행 제거
        lines   = [ln for ln in cleaned.split('\n') if not is_noise_line(ln)]
        cleaned = '\n'.join(lines)

        # search_tags 추가
        doc.page_content        = enrich_tricare_text(cleaned, fname)
        doc.metadata['source_file'] = fname
        doc.metadata['location']    = loc
        kept.append(doc)

    all_pdf_docs.extend(kept)
    print(f'  ✅ {fname} [{loc}]: {len(docs)}페이지 중 {len(kept)}페이지 사용')

print(f'\n📄 총 사용 페이지: {len(all_pdf_docs)}  (필터로 제외된 페이지: {skipped_pages})')
print(f'\n--- 샘플 메타데이터 ---')
print(all_pdf_docs[0].metadata)
print(f'\n--- 샘플 page_content (앞 400자) ---')
print(all_pdf_docs[0].page_content[:400])


  ✅ Costs_Fees.pdf [BOTH]: 4페이지 중 3페이지 사용
  ✅ Overseas_HB(해외 프로그램 안내서).pdf [OCONUS]: 16페이지 중 16페이지 사용
  ✅ Pharmacy_HB(tricare 약국 프로그램 안내서).pdf [BOTH]: 32페이지 중 14페이지 사용
  ✅ NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf [BOTH]: 16페이지 중 4페이지 사용
  ✅ TFL_HB(평생 트라이케어).pdf [BOTH]: 44페이지 중 20페이지 사용
  ✅ TRICARE_ADDP_HB_FINAL_508c(현역 군인 치과 프로그램 안내서).pdf [BOTH]: 24페이지 중 16페이지 사용
  ✅ ADDP_Brochure_FINAL_122624_508c.pdf [BOTH]: 4페이지 중 3페이지 사용
  ✅ Maternity_Br (1).pdf [BOTH]: 4페이지 중 4페이지 사용
  ✅ Retiring_NGR_Br.pdf [BOTH]: 4페이지 중 3페이지 사용
  ✅ QLEs_FS (2).pdf [BOTH]: 4페이지 중 3페이지 사용

📄 총 사용 페이지: 86  (필터로 제외된 페이지: 66)

--- 샘플 메타데이터 ---
{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2026-01-06T16:05:56-05:00', 'source': 'C:\\SKN_24\\3차 프로젝트\\Care-Insure-Bot\\data\\Costs_Fees.pdf', 'file_path': 'C:\\SKN_24\\3차 프로젝트\\Care-Insure-Bot\\data\\Costs_Fees.pdf', 'total_pages': 4, 'format': 'PDF 1.6', 'title': 'TRICARE 2026 Costs and Fees Sheet', 'author': 'TRICARE Co

---
## Section 5 · PDF 청킹 — 단락 단위 분리

### 개요
- [변경] RecursiveCharacterTextSplitter → **단락(`\n\n`) 단위 직접 분리**로 교체
  - 청크 경계가 문장 중간에서 잘리는 문제 방지
  - 80자 이상 단락만 Document로 생성 (너무 짧은 단락 제거)
- RecursiveCharacterTextSplitter는 CSV 청킹에만 재사용됨 (Section 6)
- 단락 분리 후에도 `source_file`, `location` 메타데이터는 그대로 유지됨


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document as LCDocument

# CSV 청킹용 text_splitter는 그대로 유지 (Section 6에서 재사용)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', '']
)

# PDF는 단락(\n\n) 단위 직접 분리 — 문장 중간 절단 방지
# 80자 미만 단락은 헤더·페이지 번호 등 노이즈일 가능성이 높으므로 제거
MIN_CHUNK_CHARS = 80

pdf_chunks = []
for doc in all_pdf_docs:
    # search_tags 블록은 분리하지 않고 단락 분리 전에 제거 후 각 청크에 다시 붙임
    # (search_tags가 모든 청크에 포함되어야 유사도 향상 효과가 있음)
    full_text = doc.page_content
    if '[search_tags]' in full_text:
        main_text, tags_block = full_text.split('[search_tags]', 1)
        tags_block = '[search_tags]' + tags_block
    else:
        main_text  = full_text
        tags_block = ''

    paragraphs = [p.strip() for p in main_text.split('\n\n') if len(p.strip()) >= MIN_CHUNK_CHARS]
    for para in paragraphs:
        chunk_text = para + ('\n\n' + tags_block if tags_block else '')
        pdf_chunks.append(LCDocument(
            page_content=chunk_text,
            metadata=doc.metadata.copy()
        ))

print(f'분할 결과: {len(all_pdf_docs)}페이지 → {len(pdf_chunks)}개 청크')
print(f'(단락 단위 분리, {MIN_CHUNK_CHARS}자 미만 제거)')
print(f'\n--- 청크 샘플 ---')
print(pdf_chunks[0].page_content[:400])
print(f'\nMetadata: {pdf_chunks[0].metadata}')


분할 결과: 86페이지 → 86개 청크
(단락 단위 분리, 80자 미만 제거)

--- 청크 샘플 ---
TRICARE® 2026 Costs and Fees
TRICARE For Life, survivors, and medically retired individuals, visit www.tricare.mil/comparecosts. 
• You’re in Group A if your initial enlistment or appointment or that of your uniformed services sponsor began before Jan. 1, 2018. 
• You’re in Group B if your initial enlistment or appointment or that of your uniformed services sponsor began on or after Jan. 1, 2018.


Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2026-01-06T16:05:56-05:00', 'source': 'C:\\SKN_24\\3차 프로젝트\\Care-Insure-Bot\\data\\Costs_Fees.pdf', 'file_path': 'C:\\SKN_24\\3차 프로젝트\\Care-Insure-Bot\\data\\Costs_Fees.pdf', 'total_pages': 4, 'format': 'PDF 1.6', 'title': 'TRICARE 2026 Costs and Fees Sheet', 'author': 'TRICARE Communications', 'subject': '', 'keywords': '', 'moddate': '2026-01-28T17:23:05-05:00', 'trapped': '', 'modDate': "D:20260128172305-05'00'", 'creati

---
## Section 6 · CSV 로드 — 텍스트 벡터 저장소용 3종

### 개요
- integrated2와 동일한 방식으로 유지함
- `CSVLoader` 대신 직접 변환 방식을 사용함
  - 청킹 후에도 서비스명·항목명이 각 청크에 남아 출처 추적이 가능함

### 3종 파일별 변환 방식
| 파일 | page_content 구성 | metadata 키 |
|---|---|---|
| `mental_health_services.csv` | `서비스: {서비스}\n내용: {내용}` | `source`, `서비스명`, `type` |
| `TricarePlans.csv` | `항목: {Plans}\n플랜명: {플랜명}\n내용: {내용}` | `source`, `plan_name`, `type` |
| `tricare_exclusions.csv` | `제외항목: {name}\n내용: {content}` | `source`, `exclusion_name`, `type`, `url` |


In [ ]:
import pandas as pd
from langchain_core.documents import Document

# mental_health_services.csv 
# 정신건강 서비스별 보장 내용
# 서비스명을 page_content에 직접 포함 → 청킹 후에도 서비스명이 남아 출처 추적 가능
df_mental  = pd.read_csv(CSV_MENTAL, encoding='utf-8-sig')
mental_docs = []
for _, row in df_mental.iterrows():
    mental_docs.append(Document(
        page_content=f"서비스: {row['서비스']}\n내용: {row['내용']}",
        metadata={
            'source':   'mental_health_services.csv',
            '서비스명': row['서비스'],
            'type':     'mental_health',
            'location': 'BOTH',  # CSV는 플랜 공통 내용이므로 BOTH
        }
    ))

mental_chunks = text_splitter.split_documents(mental_docs)
print(f'📋 mental_health_services: {len(df_mental)}행 → {len(mental_chunks)}청크')


📋 mental_health_services: 8행 → 12청크


In [ ]:
# TricarePlans.csv 
# 플랜별 특징 비교 (컬럼 구조: Plans 항목명 + 각 플랜명)
# 플랜별로 항목-내용 쌍을 Document로 변환 → 플랜 비교 질문에 검색되도록 함
df_plans  = pd.read_csv(CSV_PLANS, encoding='utf-8-sig')
plan_cols = [c for c in df_plans.columns if c != 'Plans']
plans_docs = []
for _, row in df_plans.iterrows():
    topic = row['Plans']
    for plan_name in plan_cols:
        content = str(row[plan_name]).strip()
        if not content or content == 'nan':
            continue
        plans_docs.append(Document(
            page_content=f"항목: {topic}\n플랜명: {plan_name}\n내용: {content}",
            metadata={
                'source':    'TricarePlans.csv',
                'plan_name': plan_name,
                'type':      'plan_info',
                'location':  'BOTH',  # 플랜 정보는 공통
            }
        ))

plans_chunks = text_splitter.split_documents(plans_docs)
print(f'📋 TricarePlans: {len(df_plans)}행 × {len(plan_cols)}플랜 → {len(plans_chunks)}청크')


📋 TricarePlans: 10행 × 16플랜 → 164청크


In [10]:
# ── tricare_exclusions.csv ──
# TRICARE 보장 제외 항목 (tricare.mil 크롤링 → CSV 저장본)
# url은 metadata에만 저장하여 LLM 컨텍스트에는 노출하지 않음
df_excl = pd.read_csv(CSV_EXCLUSIONS, encoding='utf-8-sig')
exclusion_docs = []
for _, row in df_excl.iterrows():
    exclusion_docs.append(Document(
        page_content=f"제외항목: {row['name']}\n내용: {row['content']}",
        metadata={
            'source':         'tricare_exclusions.csv',
            'exclusion_name': row['name'],
            'type':           'exclusion',
            'url':            row['url'],
            'location':       'BOTH',  # 제외 항목은 플랜 공통
        }
    ))

exclusion_chunks = text_splitter.split_documents(exclusion_docs)
print(f'📋 tricare_exclusions: {len(df_excl)}행 → {len(exclusion_chunks)}청크')


📋 tricare_exclusions: 55행 → 64청크


---
## Section 7 · 텍스트+CSV 통합 벡터 저장소 생성 (`chroma_db`)

### 개요
- PDF 텍스트 + CSV 3종(정신건강 / 플랜 정보 / 제외 항목)을 하나의 저장소에 통합함
- Retriever가 소스 구분 없이 가장 관련 있는 청크를 한 번에 검색함
- `metadata.source`와 `metadata.type`으로 청크 출처는 항상 추적 가능함

### 통합 청크 구성
| 소스 | 청크 변수 |
|---|---|
| PDF 텍스트 15종 | `pdf_chunks` |
| mental_health_services.csv | `mental_chunks` |
| TricarePlans.csv | `plans_chunks` |
| tricare_exclusions.csv | `exclusion_chunks` |


In [11]:
from langchain_chroma import Chroma

# PDF + CSV 3종 모두 합산
all_text_chunks = pdf_chunks + mental_chunks + plans_chunks + exclusion_chunks

print(f'📦 청크 합산')
print(f'PDF 텍스트 ({len(RAG_PDF_DOCS)}종):      {len(pdf_chunks)}개')
print(f'mental_health_services: {len(mental_chunks)}개')
print(f'TricarePlans:           {len(plans_chunks)}개')
print(f'tricare_exclusions:     {len(exclusion_chunks)}개')
print(f'합계:                   {len(all_text_chunks)}개')
print('⏳ 벡터 저장소 생성 중...')

# PERSIST_TEXT / COLLECTION_TEXT는 Section 1에서 정의한 상수
vector_store = Chroma.from_documents(
    documents=all_text_chunks,
    embedding=embedding_model,
    collection_name=COLLECTION_TEXT,
    persist_directory=PERSIST_TEXT
)

print(f'✅ 텍스트 벡터 저장소 생성 완료: {vector_store._collection.count()}개 벡터')


📦 청크 합산
PDF 텍스트 (10종):      86개
mental_health_services: 12개
TricarePlans:           164개
tricare_exclusions:     64개
합계:                   326개
⏳ 벡터 저장소 생성 중...
✅ 텍스트 벡터 저장소 생성 완료: 326개 벡터


---
## Section 8 · PDF 표 추출 — pdfplumber + `Health_Plan_Costs.csv`

### 개요
- [변경] 기존 `Costs_Fees.pdf` 단일 대상 → `TABLE_PDF_DOCS` 리스트 순회 방식으로 확장함
  - `Pharmacy_HB.pdf`, `NGR_HB.pdf`, `TFL_HB.pdf`, `Plans_Overview_FS.pdf` 표 추가 수록
- `pdfplumber`는 표의 셀 좌표를 인식해 `행 | 열` 구조를 보존함
- 표는 텍스트 벡터 저장소와 분리해 별도 저장함
  - 비용 질문 시 텍스트 청크가 섞이면 노이즈가 될 수 있음
- `Health_Plan_Costs.csv`는 행 자체가 비용 단위로 구성되어 있어 추가 청킹 불필요함


In [12]:
import pdfplumber

def extract_tables_from_pdf(pdf_path: str) -> list:
    """
    pdfplumber로 PDF 전체에서 표를 추출함.
    [변경] normalize_cell() 적용 — ✓/✗ → Covered/Not covered 정규화
    [변경] 빈 행 제거 — 모든 셀이 N/A인 행은 노이즈이므로 제외
    각 셀은 ' | '로 구분되어 LLM이 행/열 관계를 파악할 수 있음.
    source_file 메타데이터로 어느 PDF에서 왔는지 팀 통합 시 구분 가능함.
    """
    table_docs = []
    file_name  = os.path.basename(pdf_path)
    loc        = LOCATION_MAP.get(file_name, 'BOTH')
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for table_idx, table in enumerate(tables):
                if not table or len(table) < 2:
                    continue
                lines = []
                for row in table:
                    # normalize_cell 적용 — ✓/✗/빈칸을 텍스트로 변환
                    cleaned_row = [
                        normalize_cell(str(cell).strip().replace('\n', ' ') if cell else '')
                        for cell in row
                    ]
                    # 모든 셀이 N/A인 행은 빈 행이므로 제거
                    if all(c == 'N/A' for c in cleaned_row):
                        continue
                    lines.append(' | '.join(cleaned_row))
                if not lines:
                    continue
                table_docs.append(LCDocument(
                    page_content=f'[TRICARE 비용표 | {file_name} | p.{page_num + 1}]\n' + '\n'.join(lines),
                    metadata={
                        'source':      pdf_path,
                        'source_file': file_name,
                        'page':        page_num + 1,
                        'type':        'cost_table',
                        'table_idx':   table_idx + 1,
                        'location':    loc,
                    }
                ))
    return table_docs

# TABLE_PDF_DOCS 리스트를 순회하며 일괄 표 추출
table_docs_pdf = []
for pdf_path in TABLE_PDF_DOCS:
    docs = extract_tables_from_pdf(pdf_path)
    table_docs_pdf.extend(docs)
    print(f'  📊 {os.path.basename(pdf_path)}: {len(docs)}개 표 추출')

print(f'\n📊 PDF 표 합계: {len(table_docs_pdf)}개')
print('\n--- PDF 표 샘플 ---')
if table_docs_pdf:
    print(table_docs_pdf[0].page_content[:300])


  📊 Costs_Fees.pdf: 8개 표 추출
  📊 Pharmacy_HB(tricare 약국 프로그램 안내서).pdf: 8개 표 추출
  📊 NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf: 14개 표 추출
  📊 TFL_HB(평생 트라이케어).pdf: 14개 표 추출

📊 PDF 표 합계: 44개

--- PDF 표 샘플 ---
[TRICARE 비용표 | Costs_Fees.pdf | p.1]
ADSMs, ADFMs, and transitional survivors | N/A | N/A
Covered service | Group A | Group B
All covered services | $0 | $0
Retirees, their family members, and all others | N/A | N/A
Covered service | Group A | Group B
Preventive care visit | $0 | $0
Primary care out


In [13]:
# Health_Plan_Costs.csv — 비용 전용 저장소용
# 탭(플랜·그룹 정보) + 비용 유형 + 플랜 1/2 수치를 하나의 page_content로 구성
# pdfplumber 표와 동일한 저장소에 넣어 비용 질문의 검색 커버리지를 높임
df_costs = pd.read_csv(CSV_COSTS, encoding='utf-8-sig')
cost_docs = []
for _, row in df_costs.iterrows():
    tab       = str(row['탭']).strip()
    cost_type = str(row['비용 유형']).strip()
    plan1     = str(row['플랜 1']).strip()
    plan2     = str(row['플랜 2']).strip() if str(row['플랜 2']) != 'nan' else ''

    content = f"플랜/그룹: {tab}\n비용 유형: {cost_type}\n플랜 1: {plan1}"
    if plan2:
        content += f"\n플랜 2: {plan2}"

    cost_docs.append(Document(
        page_content=content,
        metadata={
            'source': 'Health_Plan_Costs.csv',
            'cost_type': cost_type,
            'type': 'cost_table'
        }
    ))

# Health_Plan_Costs는 행 자체가 비용 단위로 구성되어 있어 추가 청킹 불필요함
print(f'📊 Health_Plan_Costs: {len(df_costs)}행 → {len(cost_docs)}개 Document')
print('\n--- CSV 비용 샘플 ---')
print(cost_docs[0].page_content)
print(cost_docs[1].page_content)
print(cost_docs[2].page_content)
print(cost_docs[3].page_content)
print(cost_docs[4].page_content)


📊 Health_Plan_Costs: 234행 → 234개 Document

--- CSV 비용 샘플 ---
플랜/그룹: TRICARE Prime & TRICARE Select Active duty family members & transitional survivors Group A
비용 유형: Enrollment fee (annual)
플랜 1: $0
플랜 2: $0
플랜/그룹: TRICARE Prime & TRICARE Select Active duty family members & transitional survivors Group A
비용 유형: Deductible (annual)
플랜 1: $0
플랜 2: E-1 through E-4: $50 per individual $100 per family E-5 & above: $150 per individual $300 per family
플랜/그룹: TRICARE Prime & TRICARE Select Active duty family members & transitional survivors Group A
비용 유형: Catastrophic cap (annual)
플랜 1: $1,000 per family
플랜 2: $1,000 per family
플랜/그룹: TRICARE Prime & TRICARE Select Active duty family members & transitional survivors Group A
비용 유형: Clinical preventive services
플랜 1: Network: $0 Non-network: Point-of-service fees
플랜 2: Network and Non-network: $0
플랜/그룹: TRICARE Prime & TRICARE Select Active duty family members & transitional survivors Group A
비용 유형: Outpatient visit — primary care
플랜 1: Network:

---
## Section 9 · 표 전용 벡터 저장소 생성 (`chroma_db2`)

### 개요
- PDF 표 + `Health_Plan_Costs.csv`를 하나의 표 전용 저장소에 통합함
- `k=3`: 표 데이터는 정보 밀도가 높아 3개면 충분함
- 비용 질문 시 텍스트 청크가 섞이지 않아 수치 답변 정확도가 높아짐

### 재실행 시 벡터 저장소 불러오는 방법
```python
vector_store = Chroma(
    collection_name='tricare_rag',
    embedding_function=embedding_model,
    persist_directory='./chroma_db'
)
table_vector_store = Chroma(
    collection_name='tricare_cost_tables',
    embedding_function=embedding_model,
    persist_directory='./chroma_db2'
)
```


In [14]:
# PDF 표 + CSV 비용 Document 합산
all_table_docs = table_docs_pdf + cost_docs

print(f'📦 표 데이터 합산')
print(f'   PDF 표 (pdfplumber, {len(TABLE_PDF_DOCS)}종): {len(table_docs_pdf)}개')
print(f'   Health_Plan_Costs.csv:              {len(cost_docs)}개')
print(f'   합계:                               {len(all_table_docs)}개')
print('⏳ 표 전용 벡터 저장소 생성 중...')

# PERSIST_TABLE / COLLECTION_TABLE은 Section 1에서 정의한 상수
table_vector_store = Chroma.from_documents(
    documents=all_table_docs,
    embedding=embedding_model,
    collection_name=COLLECTION_TABLE,
    persist_directory=PERSIST_TABLE
)

print(f'✅ 표 전용 벡터 저장소 생성 완료: {table_vector_store._collection.count()}개 벡터')


📦 표 데이터 합산
   PDF 표 (pdfplumber, 4종): 44개
   Health_Plan_Costs.csv:              234개
   합계:                               278개
⏳ 표 전용 벡터 저장소 생성 중...
✅ 표 전용 벡터 저장소 생성 완료: 278개 벡터


---
## Section 10 · Retriever 설정 — Hybrid + Rerank

### [integrated5 → v3 변경 사항]
- 기존: MMR 단일 검색
- 변경: **BM25 + MMR Hybrid → CrossEncoder Rerank** 파이프라인으로 교체
- 실험 파일(`tricare_retriever_exp.ipynb`) EX-5 정량 평가 결과 반영

### 실험 결과 (반영 근거)
| 방식 | Hit Rate | MRR | Faithfulness |
|---|---|---|---|
| A. MMR (기존) | 1.00 | 0.667 | 0.920 |
| D. Hybrid+Rerank (신규) | 1.00 | **0.867** | **1.000** |

### 파이프라인 흐름
```
질문
 ↓
BM25(k=20) + MMR(k=20) → Hybrid(20개 후보)
 ↓
CrossEncoder Rerank → 상위 6개 선별
 ↓
LLM 컨텍스트
```


In [ ]:
# [integrated5 → v3 변경] Section 10 전체 교체
# 기존: MMR 단일 retriever
# 변경: BM25 + Hybrid + CrossEncoder Rerank 파이프라인
# 실험 파일(tricare_retriever_exp.ipynb) EX-5 결과 기반

from langchain_openai import ChatOpenAI
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import CrossEncoder
import json as _json

#  normalize_question 
# LLM이 질문의 intent / region / 영어쿼리를 추출
# 한국어 질문 → 영어 변환 → 벡터 검색 정확도 향상
# gpt-4o-mini 호출 비용 발생 → RAG 검색 단계에서만 호출
_norm_model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

NORMALIZE_PROMPT = """
You are a TRICARE insurance query analyzer.
Given the user's question, extract:
1. intent: one of [coverage, eligibility, cost, pharmacy, dental, overseas, general]
2. region: one of [CONUS, OCONUS, korea, unknown]
3. english_query: rewrite the question in clear English (for vector search)

Respond ONLY in JSON format:
{{"intent": "...", "region": "...", "english_query": "..."}}

Question: {question}
"""

def normalize_question(question: str) -> dict:
    """질문에서 intent / region / 영어쿼리 추출"""
    try:
        resp = _norm_model.invoke(NORMALIZE_PROMPT.format(question=question))
        return _json.loads(resp.content)
    except Exception:
        return {'intent': 'general', 'region': 'unknown', 'english_query': question}


#  BM25Retriever 
# [신규 추가] 키워드 빈도 기반 검색
# 'Group A', 'copay' 같은 정확한 용어 매칭에 강함
# all_text_chunks는 Section 7에서 생성된 통합 청크 리스트
bm25_retriever = BM25Retriever.from_documents(
    all_text_chunks,
    k=6
)

#  hybrid_retrieve 
# [신규 추가] BM25 + MMR 결과를 RRF 방식으로 합산
# EnsembleRetriever 버전 호환 문제로 직접 구현
# bm25_weight + vec_weight = 1.0 이 되도록 설정
def hybrid_retrieve(question: str, bm25_ret, vec_ret,
                    bm25_weight: float = 0.4, vec_weight: float = 0.6,
                    k: int = 6) -> list:
    """BM25 + 벡터 검색 결과를 순위 기반 가중치로 합산"""
    bm25_docs = bm25_ret.invoke(question)
    vec_docs  = vec_ret.invoke(question)

    scores  = {}
    doc_map = {}

    for rank, doc in enumerate(bm25_docs):
        key = doc.page_content[:100]
        scores[key]  = scores.get(key, 0) + bm25_weight * (1 / (rank + 1))
        doc_map[key] = doc

    for rank, doc in enumerate(vec_docs):
        key = doc.page_content[:100]
        scores[key]  = scores.get(key, 0) + vec_weight * (1 / (rank + 1))
        doc_map[key] = doc

    sorted_keys = sorted(scores, key=lambda x: scores[x], reverse=True)
    return [doc_map[key] for key in sorted_keys[:k]]


# hybrid_retrieve_wide 
# [신규 추가] Rerank용 넓은 후보 수집 (k=20)
# BM25(k=20) + MMR(k=20) 합산 → 최대 20개 후보 반환
def hybrid_retrieve_wide(question: str, k: int = 20) -> list:
    """Rerank 전 단계: 넓은 범위로 후보 문서 수집"""
    bm25_wide = BM25Retriever.from_documents(all_text_chunks, k=k)
    vec_wide  = vector_store.as_retriever(
        search_type='mmr',
        search_kwargs={'k': k, 'fetch_k': 40}
    )
    return hybrid_retrieve(question, bm25_wide, vec_wide, k=k)


# CrossEncoder Reranker 
# [신규 추가] 1차 검색 후보를 질문-문서 쌍으로 재점수
# BAAI/bge-reranker-v2-m3: 한국어 포함 다국어 지원
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')

def rerank_docs(question: str, docs: list, top_k: int = 6) -> list:
    """
    CrossEncoder로 docs 재점수 후 상위 top_k개 반환
    search_tags 블록은 제거 후 점수 매김 (노이즈 방지)
    """
    if not docs:
        return docs
    pairs = []
    for doc in docs:
        content = doc.page_content
        if '[search_tags]' in content:
            content = content.split('[search_tags]')[0].strip()
        pairs.append((question, content))

    scores = reranker.predict(pairs)
    scored = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored[:top_k]]


# 최종 Retriever 설정 
# [변경] 기존 MMR retriever → Hybrid+Rerank 방식으로 교체
# retriever: 텍스트 검색용 (make_rag_chain 호환성 유지)
# table_retriever: 표 전용 (비용 질문용, MMR 유지)
retriever = vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 6, 'fetch_k': 20}
)
table_retriever = table_vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 3, 'fetch_k': 10}
)

#  동작 확인 
test_q = 'TRICARE Prime 한국 overseas copay Group A'

# Hybrid+Rerank 결과 확인
candidates = hybrid_retrieve_wide(test_q)
reranked   = rerank_docs(test_q, candidates, top_k=6)

print(f'🔍 Hybrid+Rerank 테스트: {len(candidates)}개 후보 → {len(reranked)}개 선별')
for i, d in enumerate(reranked):
    src  = d.metadata.get('source_file', d.metadata.get('source', '?'))
    loc  = d.metadata.get('location', '?')
    page = d.metadata.get('page', '')
    print(f'  [{i+1}] {src} | loc={loc} | p.{page}')
    print(f'       {d.page_content[:70].replace(chr(10), " ")}...')


🔍 Hybrid+Rerank 테스트: 20개 후보 → 6개 선별
  [1] Costs_Fees.pdf | loc=BOTH | p.0
       TRICARE® 2026 Costs and Fees TRICARE For Life, survivors, and medicall...
  [2] Pharmacy_HB(tricare 약국 프로그램 안내서).pdf | loc=BOTH | p.19
       Costs by Beneficiary Group  Cost-shares, deductibles, and catastrophic...
  [3] Overseas_HB(해외 프로그램 안내서).pdf | loc=OCONUS | p.3
       YOUR TRICARE OPTIONS BY SPONSOR STATUS   Your TRICARE health plan opti...
  [4] Overseas_HB(해외 프로그램 안내서).pdf | loc=OCONUS | p.12
       TRICARE PHARMACY PROGRAM  The TRICARE Pharmacy Program provides prescr...
  [5] TricarePlans.csv | loc=BOTH | p.
       항목: What is it? 플랜명: TRICARE Prime Overseas 내용: A managed care option ...
  [6] TricarePlans.csv | loc=BOTH | p.
       항목: What is it? 플랜명: TRICARE Prime Remote Overseas 내용: A managed care ...


---
## Section 11 · LLM + RAG 체인 구성

### 개요
- integrated2와 동일하게 유지함 (팀원 통일 사항)
- `gpt-4o-mini`, `temperature=0.1`: 보험 정보 오답은 실제 피해로 이어질 수 있어
  낮은 temperature로 문서 기반 답변을 유도함
- `detect_language()`: 유니코드 범위로 한/중/일을 판별하고 나머지는 영어로 처리함
  - 팀원 A(rag_utils.py)와 동일한 방식으로 통일함
- `make_rag_chain()`: 질문마다 언어를 감지해 프롬프트에 `answer_language`를 f-string으로 주입함
  - `retriever`를 인자로 받아 텍스트용 / 표 전용 체인 모두 이 함수 하나로 생성 가능함
- `format_docs()`: 출처 정보(`[source, p.페이지]`)를 청크 앞에 붙여 반환함
  - 팀원 RAG_TEMPLATE의 출처 인용 형식과 통일함

  **⬇️⬇️⬇️⬇️ 수정함***

  ## Section 11 · LLM + RAG 체인 구성

### 개요
- `gpt-4o-mini`, `temperature=0.1`: 보험 정보 오답은 실제 피해로 이어질 수 있어
  낮은 temperature로 문서 기반 답변을 유도함
- `detect_language()`: 유니코드 범위로 한/중/일을 판별하고 나머지는 영어로 처리함
  - 팀원 A(rag_utils.py)와 동일한 방식으로 통일함
- `make_rag_chain()`: 질문마다 언어를 감지해 프롬프트에 `answer_language`를 f-string으로 주입함
  - `retriever`를 인자로 받아 텍스트용 / 표 전용 체인 모두 이 함수 하나로 생성 가능함
- `format_docs()`: 출처 정보(`[source, p.페이지]`)를 청크 앞에 붙여 반환함
  - 팀원 RAG_TEMPLATE의 출처 인용 형식과 통일함
- [변경] OCONUS 컨텍스트 추가: 주한미군·한국 거주 수혜자 대상 시스템임을 프롬프트에 명시함
  - 한국에서는 트라이케어가 1차 지불자, Medicare 해외 적용 불가 등 핵심 규칙 주입


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

# 언어 감지 함수 (팀원 통일 방식 그대로 유지)
def detect_language(text: str) -> str:
    if any('\u4e00' <= c <= '\u9fff' for c in text):
        return 'zh'
    if any('\u3040' <= c <= '\u30ff' for c in text):
        return 'ja'
    if any('\uac00' <= c <= '\ud7a3' for c in text):
        return 'ko'
    return 'en'

LANGUAGE_NAME_MAP = {
    'ko': 'Korean',
    'en': 'English',
    'zh': 'Chinese',
    'ja': 'Japanese',
    'es': 'Spanish',
    'fr': 'French',
    'de': 'German',
    'other': "the same language as the user's question",
}

# format_docs: 출처 정보(source_file, p.페이지)를 청크 앞에 붙임
# search_tags 블록은 LLM 컨텍스트에서 제거 — LLM에게 태그 텍스트가 보이면 노이즈
def format_docs(docs):
    result = []
    for doc in docs:
        source = doc.metadata.get('source_file', doc.metadata.get('source', 'unknown'))
        page   = doc.metadata.get('page', '')
        label  = f'[{source}{", p." + str(page) if page else ""}]'
        # search_tags 블록 제거 후 LLM에 전달
        content = doc.page_content
        if '[search_tags]' in content:
            content = content.split('[search_tags]')[0].strip()
        result.append(f'{label}\n{content}')
    return '\n\n'.join(result)


def make_rag_chain(question: str, retriever, model):
    """
    [변경] normalize_question으로 영어 쿼리 추출 → mmr_retrieve 사용
    retriever 인자는 호환성 유지를 위해 그대로 받되, 내부적으로 store 직접 접근
    """
    language_code   = detect_language(question)
    answer_language = LANGUAGE_NAME_MAP.get(language_code, 'English')

    PROMPT_TEMPLATE = (
        'You are a TRICARE health benefits specialist.\n'
        'This system is designed for OCONUS beneficiaries, '
        'primarily Korean residents and USFK (주한미군) personnel.\n'
        'IMPORTANT OCONUS RULES:\n'
        '- In South Korea, TRICARE is the PRIMARY payer (not Medicare).\n'
        '- Medicare does NOT cover overseas medical expenses.\n'
        '- Overseas claims require pay-up-front then submit within 3 years.\n'
        '- Medicare Part B must be actively enrolled (not automatic) for overseas residents.\n\n'
        'Answer questions based ONLY on the provided TRICARE documents below.\n'
        'If the answer is not in the documents, '
        'say "해당 내용은 제공된 문서에서 확인되지 않습니다." in the user\'s language.\n'
        'Always mention relevant conditions such as Group A/B, plan type, or beneficiary status.\n'
        '보험 추천이나 특정 플랜을 권유하는 답변은 하지 마세요. '
        '정보 제공만 하고 최종 판단은 사용자 본인과 보험사에게 맡기세요.\n\n'
        f'IMPORTANT LANGUAGE RULE:\n'
        f'- You must answer in {answer_language}.\n'
        f'- The answer language must match the user\'s question language.\n'
        f'- Do not switch languages unless necessary for official terms. '
        f'Use term locking format: 본인부담금(Copay), 공제액(Deductible), 사전승인(Prior Authorization).\n\n'
        '[참고 문서]\n{context}\n\n'
        '[질문]\n{question}\n\n'
        '[답변]\n'
    )

    rag_prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)

    return (
        {'question': RunnablePassthrough(), 'context': retriever | format_docs}
        | rag_prompt
        | model
        | StrOutputParser()
    )


# mmr_retrieve를 활용한 make_rag_chain_v2 — normalize_question 완전 통합 버전
# Demo에서는 make_rag_chain(기존 호환) 또는 make_rag_chain_v2 중 선택 가능
def make_rag_chain_v2(question: str, store, model):
    """normalize_question + mmr_retrieve 완전 통합 버전"""
    language_code   = detect_language(question)
    answer_language = LANGUAGE_NAME_MAP.get(language_code, 'English')
    norm   = normalize_question(question)
    docs   = mmr_retrieve(question, store)
    context = format_docs(docs)

    PROMPT_TEMPLATE = (
        'You are a TRICARE health benefits specialist.\n'
        'This system is designed for OCONUS beneficiaries, '
        'primarily Korean residents and USFK (주한미군) personnel.\n'
        'IMPORTANT OCONUS RULES:\n'
        '- In South Korea, TRICARE is the PRIMARY payer (not Medicare).\n'
        '- Medicare does NOT cover overseas medical expenses.\n'
        '- Overseas claims require pay-up-front then submit within 3 years.\n'
        '- Medicare Part B must be actively enrolled (not automatic) for overseas residents.\n\n'
        f'Query intent: {norm.get("intent", "general")} | Region: {norm.get("region", "unknown")}\n\n'
        'Answer questions based ONLY on the provided TRICARE documents below.\n'
        'If the answer is not in the documents, '
        'say "해당 내용은 제공된 문서에서 확인되지 않습니다." in the user\'s language.\n'
        'Always mention relevant conditions such as Group A/B, plan type, or beneficiary status.\n'
        '보험 추천이나 특정 플랜을 권유하는 답변은 하지 마세요.\n\n'
        f'IMPORTANT LANGUAGE RULE:\n'
        f'- You must answer in {answer_language}.\n'
        f'- Use term locking: 본인부담금(Copay), 공제액(Deductible), 사전승인(Prior Authorization).\n\n'
        f'[참고 문서]\n{context}\n\n'
        f'[질문]\n{question}\n\n'
        '[답변]\n'
    )

    from langchain_core.messages import HumanMessage
    resp = model.invoke([HumanMessage(content=PROMPT_TEMPLATE)])
    return resp.content, docs


#  make_rag_chain_v3 
# [integrated5 → v3 신규 추가]
# Hybrid+Rerank 완전 통합 버전
# 실험 결과 D (Hybrid+Rerank) 방식이 최고 성능 확인 후 반영
# Hit Rate 1.0 / MRR 0.867 / Faithfulness 1.0
def make_rag_chain_v3(question: str, model):
    """
    Hybrid+Rerank 완전 통합 RAG 체인
    1. hybrid_retrieve_wide: BM25+MMR으로 20개 후보 수집
    2. rerank_docs: CrossEncoder로 상위 6개 선별
    3. LLM: 선별된 문서 기반 답변 생성
    """
    from langchain_core.messages import HumanMessage

    language_code   = detect_language(question)
    answer_language = LANGUAGE_NAME_MAP.get(language_code, 'English')

    # 1단계: Hybrid로 넓게 후보 수집
    candidates = hybrid_retrieve_wide(question)
    # 2단계: Rerank로 상위 6개 선별
    docs       = rerank_docs(question, candidates, top_k=6)
    context    = format_docs(docs)

    PROMPT_TEMPLATE = (
        'You are a TRICARE health benefits specialist.\n'
        'This system is designed for OCONUS beneficiaries, '
        'primarily Korean residents and USFK (주한미군) personnel.\n'
        'IMPORTANT OCONUS RULES:\n'
        '- In South Korea, TRICARE is the PRIMARY payer (not Medicare).\n'
        '- Medicare does NOT cover overseas medical expenses.\n'
        '- Overseas claims require pay-up-front then submit within 3 years.\n'
        '- Medicare Part B must be actively enrolled (not automatic) for overseas residents.\n\n'
        'Answer questions based ONLY on the provided TRICARE documents below.\n'
        'If the answer is not in the documents, '
        'say "해당 내용은 제공된 문서에서 확인되지 않습니다." in the user\'s language.\n'
        'Always mention relevant conditions such as Group A/B, plan type, or beneficiary status.\n'
        '보험 추천이나 특정 플랜을 권유하는 답변은 하지 마세요.\n\n'
        f'IMPORTANT LANGUAGE RULE:\n'
        f'- You must answer in {answer_language}.\n'
        f'- Use term locking: 본인부담금(Copay), 공제액(Deductible), 사전승인(Prior Authorization).\n\n'
        f'[참고 문서]\n{context}\n\n'
        f'[질문]\n{question}\n\n'
        '[답변]\n'
    )

    resp = model.invoke([HumanMessage(content=PROMPT_TEMPLATE)])
    return resp.content, docs  # (답변, 검색된 docs) 반환


print('✅ LLM + RAG 체인 구성 완료 (v3 — Hybrid+Rerank 통합)')
print('   make_rag_chain(q, retriever, model)    → 기존 호환 버전 (MMR)')
print('   make_rag_chain_v2(q, store, model)     → normalize_question + MMR')
print('   make_rag_chain_v3(q, model)            → Hybrid+Rerank 최고 성능 버전')


✅ LLM + RAG 체인 구성 완료 (v3 — Hybrid+Rerank 통합)
   make_rag_chain(q, retriever, model)    → 기존 호환 버전 (MMR)
   make_rag_chain_v2(q, store, model)     → normalize_question + MMR
   make_rag_chain_v3(q, model)            → Hybrid+Rerank 최고 성능 버전


---
## Section 12 · Demo 1 — 일반 LLM vs RAG 비교

### 개요
- 같은 질문을 일반 LLM과 RAG에 각각 던져 답변 품질을 비교함
- 일반 LLM: 학습 데이터 기반으로 답변 → 세부 조건 누락 가능
- RAG: 실제 문서에서 정확한 내용을 검색 후 답변
- 출력 형식: `QUESTION → ANSWER → RETRIEVED DOCS` 순서로 통일함


In [17]:
from langchain_core.messages import HumanMessage

question = (
    'TRICARE Prime에서 정신건강 상담(Psychotherapy)은 어떻게 보장되나요? '
    '세션 종류별(개인/가족/그룹) 시간 제한과 주의사항도 알려주세요.'
)

print('=' * 100)
print('QUESTION:', question)

print('\n❓ [Demo 1 - 일반 LLM 답변] — 문서 없이 직접 응답')
print('-' * 65)
plain_response = model.invoke([HumanMessage(content=question)])
print(plain_response.content)
print('\n⚠️  주의: 위 답변은 학습 데이터 기반이며 실제 문서와 다를 수 있음')

print('\n✅ [Demo 1 - RAG 답변] — 문서 기반 응답')
print('-' * 65)
# [v3 변경] make_rag_chain → make_rag_chain_v3 (Hybrid+Rerank)
rag_answer, rag_docs = make_rag_chain_v3(question, model)
# make_rag_chain_v3은 (답변, docs) 튜플 반환 — 이미 위에서 처리됨

print('=== ANSWER ===')
print(rag_answer)

print('\n=== RETRIEVED DOCS ===')
for doc in retriever.invoke(question):
    print(doc.metadata)


QUESTION: TRICARE Prime에서 정신건강 상담(Psychotherapy)은 어떻게 보장되나요? 세션 종류별(개인/가족/그룹) 시간 제한과 주의사항도 알려주세요.

❓ [Demo 1 - 일반 LLM 답변] — 문서 없이 직접 응답
-----------------------------------------------------------------
TRICARE Prime에서 정신건강 상담(Psychotherapy)은 다양한 형태로 제공되며, 보장 내용은 다음과 같습니다.

### 보장 내용
1. **개인 상담**: 개인 상담은 일반적으로 1:1 세션으로 진행되며, TRICARE Prime에서 보장됩니다.
2. **가족 상담**: 가족 상담은 가족 구성원이 함께 참여하는 세션으로, TRICARE Prime에서 보장됩니다.
3. **그룹 상담**: 여러 사람이 함께 참여하는 그룹 상담도 TRICARE Prime에서 보장됩니다.

### 시간 제한
- **개인 상담**: 일반적으로 세션 당 50분에서 60분 정도 진행됩니다. 연간 세션 수는 필요에 따라 다르지만, 보통 20회까지 보장됩니다.
- **가족 상담**: 가족 상담도 개인 상담과 유사하게 50분에서 60분 정도 진행되며, 연간 세션 수는 필요에 따라 다릅니다.
- **그룹 상담**: 그룹 상담은 보통 1시간에서 1시간 30분 정도 진행되며, 그룹의 크기와 형식에 따라 다를 수 있습니다.

### 주의사항
- **사전 승인**: 특정 치료나 세션 수에 대해 사전 승인이 필요할 수 있습니다. 따라서 치료를 시작하기 전에 TRICARE에 문의하여 필요한 절차를 확인하는 것이 중요합니다.
- **네트워크 제공자 이용**: TRICARE Prime은 네트워크 내의 제공자를 이용할 때 더 많은 혜택을 제공합니다. 네트워크 외의 제공자를 이용할 경우 추가 비용이 발생할 수 있습니다.
- **정신건강 진단**: 상담을 받기 위해서는 정신건강 전문가에 의해 진단이 필요할 수 있습니다. 진단이 있어야 TRICA

---
## Section 13 · Demo 2 — 텍스트 RAG Q&A

### 개요
- `retriever`(텍스트+CSV 통합)를 사용해 다양한 질문 유형을 테스트함
- 신규 추가 PDF에서 답변이 검색되는지 확인하는 용도로도 활용 가능함
- 출력 형식: `QUESTION → === ANSWER === → === RETRIEVED DOCS ===` 순서로 통일함


In [18]:
questions = [
    # 기존 질문
    'TRICARE Prime에서 정신건강 상담(Psychotherapy)은 어떻게 보장되나요?',
    # 신규 PDF 커버리지 확인용 질문
    'TRICARE Pharmacy Home Delivery를 이용하려면 어떻게 등록하나요?',      # Pharmacy_HB
    '국가방위군이 활성화(Activated)될 때 TRICARE 플랜은 어떻게 변경되나요?', # NGR_HB
    'TRICARE For Life에서 Medicare와 비용은 어떻게 분담되나요?',            # TFL_HB
    'TRICARE ADDP에서 크라운(Crown)은 사전승인이 필요한가요?',              # TRICARE_ADDP_HB
    '임신 중 다른 지역으로 이사할 때 TRICARE 보장은 어떻게 되나요?',         # Maternity_Br
    '65세가 되면 TRICARE Prime에서 어떤 플랜으로 전환되나요?',              # Medicare_Turning_65_Br
    'QLE(Qualifying Life Event)에 해당하는 사례는 어떤 것들이 있나요?',     # QLEs_FS
]

for q in questions:
    print('=' * 100)
    print('QUESTION:', q)

    # [v3 변경] make_rag_chain_v3 (Hybrid+Rerank) 사용
    answer, rag_docs = make_rag_chain_v3(q, model)

    print('=== ANSWER ===')
    print(answer)

    print('\n=== RETRIEVED DOCS ===')
    for doc in retriever.invoke(q):
        print(doc.metadata)
    print()


QUESTION: TRICARE Prime에서 정신건강 상담(Psychotherapy)은 어떻게 보장되나요?
=== ANSWER ===
TRICARE Prime에서 정신건강 상담(Psychotherapy)은 의학적 또는 심리적으로 필요할 경우 보장됩니다. TRICARE는 정신 건강 장애를 치료하기 위해 사무실 기반의 외래 정신치료를 포함하여 입원 및 외래 서비스에 대해 보장합니다. 개인 세션은 최대 60분까지 보장되며, 위기 상황에서의 개인 세션은 최대 120분까지 연장될 수 있습니다. 

정신건강 서비스 이용 시, 본인부담금(Copay) 및 공제액(Deductible)과 같은 비용이 발생할 수 있으며, 사전승인(Prior Authorization)이 필요할 수 있습니다.

=== RETRIEVED DOCS ===
{'source': 'mental_health_services.csv', 'location': 'BOTH', '서비스명': 'Psychotherapy', 'type': 'mental_health'}
{'location': 'BOTH', 'plan_name': 'TRICARE Prime Demo', 'type': 'plan_info', 'source': 'TricarePlans.csv'}
{'modDate': "D:20260106092543-05'00'", 'title': 'TRICARE For Life Handbook', 'keywords': 'TFL; TRICARE For Life; handbook; TRICARE; TMA; Eligibility; Medicare; FAQs; Provider; Emergency; Care; Urgent; Behavioral; Behavioral Health Care; Prior Authorization; Authorization; Coverage; Dental; Pharmacy; Prescription; Drugs; Policy; Claims; EOB; Explanation of Benefits; DEERS; Ma

---
## Section 14 · Demo 3 — 표 전용 비용 Q&A

### 개요
- `table_retriever`(표 전용)를 사용해 비용 수치를 정확하게 검색함
- 텍스트 청크가 섞이지 않아 `$26`, `$45` 같은 수치 답변이 더 정확함
- 복수 질문을 리스트로 넣어 한 번에 실행 가능함


In [19]:
cost_questions = [
    'TRICARE Prime에서 Group A 재향군인 가족의 응급실 방문 비용은 얼마인가요?',
    'TRICARE Select Group B 재향군인의 네트워크 내 전문의 외래 진료비는 얼마인가요?',
    'TRICARE Reserve Select의 월 보험료(Member only)는 얼마인가요?',
    'TRICARE For Life에서 Medicare만 보장하는 서비스 이용 시 본인 부담은 얼마인가요?', # 신규 TFL_HB 표 활용
]

for q in cost_questions:
    print('=' * 100)
    print('QUESTION:', q)

    # 질문마다 언어 감지 후 체인 생성
    table_rag_chain = make_rag_chain(q, table_retriever, model)
    answer = table_rag_chain.invoke(q)

    print('=== ANSWER ===')
    print(answer)

    print('\n=== RETRIEVED DOCS ===')
    for doc in table_retriever.invoke(q):
        print(doc.metadata)
    print()


QUESTION: TRICARE Prime에서 Group A 재향군인 가족의 응급실 방문 비용은 얼마인가요?
=== ANSWER ===
TRICARE Prime에서 Group A 재향군인 가족의 응급실 방문 비용은 $79입니다.

=== RETRIEVED DOCS ===
{'source': 'Health_Plan_Costs.csv', 'cost_type': 'Emergency visit', 'type': 'cost_table'}
{'cost_type': 'Ambulance', 'type': 'cost_table', 'source': 'Health_Plan_Costs.csv'}
{'type': 'cost_table', 'source': 'Health_Plan_Costs.csv', 'cost_type': 'Catastrophic cap (annual)'}

QUESTION: TRICARE Select Group B 재향군인의 네트워크 내 전문의 외래 진료비는 얼마인가요?
=== ANSWER ===
TRICARE Select Group B 재향군인의 네트워크 내 전문의 외래 진료비는 $52입니다.

=== RETRIEVED DOCS ===
{'cost_type': 'Outpatient visit — specialty care', 'source': 'Health_Plan_Costs.csv', 'type': 'cost_table'}
{'cost_type': 'Ambulatory surgery', 'source': 'Health_Plan_Costs.csv', 'type': 'cost_table'}
{'type': 'cost_table', 'source': 'Health_Plan_Costs.csv', 'cost_type': 'Ambulance'}

QUESTION: TRICARE Reserve Select의 월 보험료(Member only)는 얼마인가요?
=== ANSWER ===
TRICARE Reserve Select의 월 보험료(Member only)는 $57.88입

---
## Section 15 · 정리 및 다음 단계

### 데이터 처리 전략 요약
| 데이터 | 처리 방식 | 저장소 |
|---|---|---|
| PDF 텍스트 15종 | PyMuPDFLoader → 청킹 | `chroma_db` |
| PDF 표 5종 | pdfplumber → Document | `chroma_db2` |
| mental_health_services.csv | 직접 row 변환 → 청킹 | `chroma_db` |
| TricarePlans.csv | 항목×플랜 쌍으로 변환 → 청킹 | `chroma_db` |
| tricare_exclusions.csv | 직접 row 변환 → 청킹 | `chroma_db` |
| Health_Plan_Costs.csv | 직접 row 변환 (청킹 불필요) | `chroma_db2` |

### Demo 결과 요약
| | 일반 LLM | RAG 텍스트 | RAG 표 |
|---|---|---|---|
| 데이터 출처 | 학습 데이터 | PDF 15종 + CSV 3종 | PDF 표 5종 + 비용 CSV |
| 정확도 | 낮음 | 높음 | 매우 높음 (수치) |
| 다국어 답변 | 가능 | ✅ 언어 감지 + 프롬프트 주입 | ✅ 언어 감지 + 프롬프트 주입 |
| 출처 표기 | 불가 | ✅ [파일명, p.페이지] | ✅ [파일명, p.페이지] |

### 향후 개선 방향
1. **메타데이터 필터링** — `type`, `plan_name`, `location` 필터를 Retriever에 추가
2. **앙상블 Retriever** — BM25(키워드) + bge-m3(의미) 혼합 검색
3. **평가 파이프라인** — RAGAS 라이브러리로 답변 품질 정량 측정
4. **통합 벡터 DB 병합** — 팀원 각자 벡터 DB를 `add_documents()`로 통합 컬렉션에 병합
